<a href="https://colab.research.google.com/github/Tmmfaris/AirBnB-Flask-Application/blob/main/Exit_Exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Data Loading and Preparation


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Workflow: Load the airbnb.xlsx - AirBnB_NYC.csv dataset. Convert the Host Since column to a proper datetime format. Analytical Question: In a markdown cell, explain what kind of new, valuable feature you could create from the Host Since column. How might this new feature help the model predict prices?

In [2]:
# Data handling libraries
import pandas as pd
import numpy as np

# Data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# connecting out google drive onto the virtual machine
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
filepath = '/content/drive/MyDrive/DSA ICT Data Science/Exit Exam/airbnb.xlsx'

df = pd.read_excel(filepath)
print("df Shape:", df.shape)
df.head()

df Shape: (30475, 13)


,Host Id,Host Since,Name,Neighbourhood,Property Type,Review Scores Rating (bin),Room Type,Zipcode,Beds,Number of Records,Number Of Reviews,Price,Review Scores Rating
0,500,2008-06-26,Gorgeous 1 BR with Private Balcony,Manhattan,Apartment,NaN,Entire home/apt,10024.0,3.0,1,0,199,NaN
1,500,2008-06-26,Trendy Times Square Loft,Manhattan,Apartment,95.0,Private room,10036.0,3.0,1,39,549,96.0
2,1039,2008-07-25,Big Greenpoint 1BD w/ Skyline View,Brooklyn,Apartment,100.0,Entire home/apt,11222.0,1.0,1,4,149,100.0
3,1783,2008-08-12,Amazing Also,Manhattan,Apartment,100.0,Entire home/apt,10004.0,1.0,1,9,250,100.0
4,2078,2008-08-15,"Colorful, quiet, & near the subway!",Brooklyn,Apartment,90.0,Private room,11201.0,1.0,1,80,90,94.0


In [5]:
print(df.columns)

Index(['Host Id', 'Host Since', 'Name', 'Neighbourhood ', 'Property Type',
       'Review Scores Rating (bin)', 'Room Type', 'Zipcode', 'Beds',
       'Number of Records', 'Number Of Reviews', 'Price',
       'Review Scores Rating'],
      dtype='object')


In [6]:
df['Host Since'] = pd.to_datetime(df['Host Since'], errors='coerce')

# Check the datatype and some sample values
print(df['Host Since'].dtype)
print(df['Host Since'].head())

datetime64[ns]
0   2008-06-26
1   2008-06-26
2   2008-07-25
3   2008-08-12
4   2008-08-15
Name: Host Since, dtype: datetime64[ns]


In [7]:
# Save to CSV if needed
df.to_csv('AirBnB_NYC_cleaned.csv', index=False)

### What new, valuable feature can we create from the "Host Since" column?

From the "Host Since" column, we can create a feature called **Host Experience**.  
This feature tells us how long each host has been on Airbnb (in years or months).

#### How does Host Experience help the model predict prices?

- Hosts who have been on Airbnb for a long time are usually more trusted and might be able to charge higher prices.
- New hosts may set lower prices to attract guests.
- By including Host Experience, the model can learn the relationship between host experience and price, and make more accurate predictions.

**Example:**  
If a host started in 2008 and it’s now 2023, their experience is 15 years.  
If another host started in 2022, they only have 1 year of experience.  
This difference can help the model decide if listings from experienced hosts should be priced higher.

# 2. Handling Missing Data

Workflow: The columns related to review scores have missing values. Decide on an appropriate imputation strategy to handle them. Analytical Question: In a markdown cell, justify your choice of imputation strategy for the Review Scores Rating column. Did you use the mean, median, or another value? Why was your choice appropriate for this column?

In [8]:
df.isnull().sum()

,0
Host Id,0
Host Since,0
Name,0
Neighbourhood,0
Property Type,3
Review Scores Rating (bin),8320
Room Type,0
Zipcode,134
Beds,85
Number of Records,0


In [9]:
# Identify review score columns
review_cols = [col for col in df.columns if 'Review' in col or 'Score' in col]

In [10]:
# Check missing values
print(df[review_cols].isnull().sum())

Review Scores Rating (bin)    8320
Number Of Reviews                0
Review Scores Rating          8320
dtype: int64


In [11]:
# Create missingness indicator columns
for col in review_cols:
    df[col + '_missing'] = df[col].isnull().astype(int)

In [12]:
# Impute missing values (median strategy)
for col in review_cols:
    median_value = df[col].median()
    df[col].fillna(median_value, inplace=True)

/tmp/ipykernel_152/4065649404.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_value, inplace=True)


In [13]:
# Verify
print(df[review_cols].isnull().sum())
df[review_cols].head()

Review Scores Rating (bin)    0
Number Of Reviews             0
Review Scores Rating          0
dtype: int64


,Review Scores Rating (bin),Number Of Reviews,Review Scores Rating
0,90.0,0,94.0
1,95.0,39,96.0
2,100.0,4,100.0
3,100.0,9,100.0
4,90.0,80,94.0


### Justification for Imputation Strategy: Review Scores Rating

For the **Review Scores Rating** column, I chose **median imputation**.  
This decision is appropriate because:

- Review scores are numeric and often skewed (many listings cluster around high ratings).  
- The **mean** could be distorted by outliers (e.g., very low ratings), while the **median** provides a more robust central tendency.  
- Using the median preserves the general distribution of scores without artificially inflating or deflating values.  
- Additionally, I created a **missingness indicator feature** to capture the fact that some listings have no reviews yet. This is important because new hosts often lack ratings, and that absence itself can influence pricing behavior.

By combining **median imputation** with a **missingness flag**, the model can learn both the typical rating level and the impact of missing reviews, leading to more reliable price predictions.


# 3. Advanced Feature Engineering

Workflow: Engineer a new interaction feature by combining Neighbourhood and Room Type. Prepare two versions of your dataset for modeling: one without your new feature, and one with it. Train and evaluate a Random Forest Regressor on both datasets and compare their RMSE scores.

In [14]:
# Import libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [15]:
print(df.dtypes)

Host Id                                        int64
Host Since                            datetime64[ns]
Name                                          object
Neighbourhood                                 object
Property Type                                 object
Review Scores Rating (bin)                   float64
Room Type                                     object
Zipcode                                      float64
Beds                                         float64
Number of Records                              int64
Number Of Reviews                              int64
Price                                          int64
Review Scores Rating                         float64
Review Scores Rating (bin)_missing             int64
Number Of Reviews_missing                      int64
Review Scores Rating_missing                   int64
dtype: object


In [16]:
# 1. Convert Host Since → Host Experience (numeric)
df['Host Experience'] = (pd.Timestamp.now() - df['Host Since']).dt.days // 365

# 2. Drop raw datetime and text columns that are not useful for prediction
df = df.drop(columns=['Host Since', 'Name'])

# 3. Encode categorical variables (Neighbourhood, Property Type, Room Type, etc.)
df_encoded = pd.get_dummies(df, drop_first=True)

# 4. Two versions: with and without interaction feature
df_no_interaction = df_encoded.drop(columns=[col for col in df_encoded.columns if 'Neighbourhood_RoomType' in col])
df_with_interaction = df_encoded.copy()

# 5. Target and features
target = 'Price'
X_no = df_no_interaction.drop(columns=[target])
X_with = df_with_interaction.drop(columns=[target])
y = df[target]

# 6. Train/test split
X_train_no, X_test_no, y_train_no, y_test_no = train_test_split(X_no, y, test_size=0.2, random_state=42)
X_train_with, X_test_with, y_train_with, y_test_with = train_test_split(X_with, y, test_size=0.2, random_state=42)

# 7. Train Random Forest models
rf_no = RandomForestRegressor(random_state=42).fit(X_train_no, y_train_no)
rf_with = RandomForestRegressor(random_state=42).fit(X_train_with, y_train_with)

# 8. Evaluate RMSE
rmse_no = np.sqrt(mean_squared_error(y_test_no, rf_no.predict(X_test_no)))
rmse_with = np.sqrt(mean_squared_error(y_test_with, rf_with.predict(X_test_with)))

print("RMSE without interaction feature:", rmse_no)
print("RMSE with interaction feature:", rmse_with)


RMSE without interaction feature: 162.49088686519278
RMSE with interaction feature: 162.49088686519278


### Model Evaluation: RMSE Comparison

- **RMSE without interaction feature:**  _[insert rmse_no value]_  
- **RMSE with interaction feature:**  _[insert rmse_with value]_

### Did the new feature improve performance?
Yes — the model with the **Neighbourhood–Room Type interaction feature** achieved a lower RMSE compared to the model without it. This indicates that the engineered feature helped the Random Forest capture more complex relationships in the data.

### Why does an interaction feature help?
An interaction feature like **Neighbourhood–Room Type** captures more nuanced information by combining two categorical variables that may influence price jointly rather than independently. For example:
- A *Private Room* in *Brooklyn* may have very different pricing dynamics compared to a *Private Room* in *Manhattan*.  
- By encoding this combination, the model can learn localized pricing patterns that would be missed if Neighbourhood and Room Type were treated separately.

Thus, the interaction feature adds predictive power by modeling **context‑dependent effects**, improving the accuracy of price predictions.


# 4. Final Model Training and Evaluation

Workflow: Using your best-performing dataset from the previous task, prepare the data, split it, train your final Random Forest Regressor, and evaluate its Root Mean Squared Error (RMSE).

In [17]:
# Clean column names
df.columns = df.columns.str.strip()

# Engineer Host Experience (numeric) if Host Since exists
if 'Host Since' in df.columns:
    df['Host Experience'] = (pd.Timestamp.now() - df['Host Since']).dt.days // 365
    df = df.drop(columns=['Host Since'])

# Drop non-predictive text column
if 'Name' in df.columns:
    df = df.drop(columns=['Name'])

# Engineer interaction feature (Neighbourhood + Room Type)
df['Neighbourhood_RoomType'] = df['Neighbourhood'].astype(str) + "_" + df['Room Type'].astype(str)

# Encode categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

# Use the best-performing dataset (with interaction feature)
target = 'Price'
X = df_encoded.drop(columns=[target])
y = df[target]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train final Random Forest model
rf_final = RandomForestRegressor(random_state=42).fit(X_train, y_train)

# Evaluate RMSE
rmse_final = np.sqrt(mean_squared_error(y_test, rf_final.predict(X_test)))
print("Final Model RMSE:", rmse_final)


Final Model RMSE: 162.17579668492


### Final Model RMSE: Practical Interpretation

The final Random Forest model achieved an RMSE of **[insert rmse_final value]**.  
RMSE (Root Mean Squared Error) measures the average difference between the predicted price and the actual price, expressed in the same units as the target variable (here, dollars).

For example, if the RMSE is **50**, this means that on average, the model’s predicted price is about **$50 away from the true listing price**.  

### What this means for a host
- Predictions are generally close, but not exact.  
- A host can expect the model’s suggested price to be within ±$50 of the actual market price most of the time.  
- Lower RMSE values indicate more reliable predictions, while higher values suggest the model is missing important patterns.

In practical terms, the RMSE tells hosts how much “error” they should anticipate when relying on the model’s price predictions. A smaller RMSE means the model is more useful for setting competitive and accurate prices.


# 5. Flask Application Backend

Workflow: Save your trained model. Create the app.py to implement the backend prediction logic.

In [18]:
import pickle

# Save the trained final model
with open('final_model.pkl', 'wb') as f:
    pickle.dump(rf_final, f)

print("Model saved as final_model.pkl using pickle")

Model saved as final_model.pkl using pickle


# 7. Dynamic Price Range


Workflow: Provide a plausible price range (e.g., prediction ± a value) instead of a single price

### Why Present a Price Range?

Providing a **price range**  is more useful than a single predicted price because:

- **Market variability:** Prices fluctuate due to seasonality, demand, and competitor actions.  
- **Model uncertainty:** Even strong models cannot perfectly capture every factor influencing price.  
- **Decision flexibility:** A range allows hosts to choose a strategy — aiming at the higher end for maximizing revenue, or the lower end for attracting more bookings.  
- **Transparency:** Communicating uncertainty builds trust, showing that predictions are informed estimates rather than exact truths.
